# Extension Workflows: Custom Problems and Custom Algorithms

This notebook shows the two most common extension paths in VAMOS: defining a problem from plain Python code and registering a lightweight custom algorithm through the plugin registry.


In [ ]:
import numpy as np

from vamos import make_problem, optimize

problem = make_problem(
    lambda x: [x[0], (1.0 + x[1]) * (1.0 - np.sqrt(x[0]))],
    n_var=2,
    n_obj=2,
    bounds=[(0.0, 1.0), (0.0, 1.0)],
    encoding="real",
)

result = optimize(problem, algorithm="nsgaii", max_evaluations=3000, pop_size=80, seed=42)
result.F[:5]


## Register a custom algorithm plugin


In [ ]:
from typing import Any, Mapping

from vamos.engine.algorithm.registry import ALGORITHMS, AlgorithmLike
from vamos.foundation.eval.population import evaluate_population_with_constraints
from vamos.foundation.kernel.backend import KernelBackend


class NotebookRandomSearch:
    def __init__(self, config: dict[str, Any], kernel: KernelBackend) -> None:
        self.config = config
        self.kernel = kernel
        self.pop_size = int(config.get("pop_size", 40))

    def run(
        self,
        problem: Any,
        termination: tuple[str, Any],
        seed: int,
        eval_strategy: Any | None = None,
        live_viz: Any | None = None,
    ) -> Mapping[str, Any]:
        rng = np.random.default_rng(seed)
        _, max_evals = termination
        X = rng.uniform(problem.xl, problem.xu, size=(self.pop_size, problem.n_var))
        F, G = evaluate_population_with_constraints(problem, X)
        return {"X": X, "F": F, "G": G, "evaluations": min(self.pop_size, int(max_evals))}


@ALGORITHMS.register("notebook_random_search")
def build_notebook_random_search(cfg: dict[str, Any], kernel: KernelBackend) -> AlgorithmLike:
    return NotebookRandomSearch(cfg, kernel)


In [ ]:
plugin_result = optimize(
    "zdt1",
    algorithm="notebook_random_search",
    max_evaluations=200,
    pop_size=40,
    seed=7,
)
plugin_result.F[:5]


## Next steps

- For reusable plugin files, see `../examples/plugins/custom_algorithm.py`.
- For higher-level extension guidance, see `../docs/topics/plugin_guide.md` and `../docs/topics/extending.md`.
